# Chapter 3 photodiode physics lab

This notebook runs entirely in your browser through JupyterLite and its Pyodide Python kernel. The functions come from the tested `photodiode.py` engine shipped with the documentation.

The models reproduce the ideal equations in Chapter 3. They do not replace measured device data.

In [ ]:
from pathlib import Path
import sys

# JupyterLite places photodiode.py beside this notebook. The fallback also
# makes the notebook convenient when opened from a repository checkout.
try:
    import photodiode
except ModuleNotFoundError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        module_dir = parent / "KrakenOS" / "Physics"
        if (module_dir / "photodiode.py").exists():
            sys.path.insert(0, str(module_dir))
            break
    import photodiode

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatLogSlider, FloatSlider, interact

plt.style.use("seaborn-v0_8-whitegrid")

## Equation 3.11: excess-carrier profile

Move the sliders to see how $D_e$, $\tau_e$, $\Delta n_p(0)$, and $G_L$ control the profile. The diffusion length is $L_e=\sqrt{D_e\tau_e}$.

In [ ]:
@interact(
    diffusion=FloatSlider(value=25, min=5, max=80, step=1, description="D (cm2/s)"),
    lifetime_us=FloatLogSlider(value=1, base=10, min=-2, max=2, step=0.05, description="tau (us)"),
    junction=FloatLogSlider(value=1e14, base=10, min=10, max=16, step=0.1, description="delta n(0)"),
    generation=FloatLogSlider(value=1e18, base=10, min=10, max=20, step=0.1, description="G_L"),
)
def plot_carriers(diffusion, lifetime_us, junction, generation):
    lifetime_s = lifetime_us * 1e-6
    length_um = photodiode.diffusion_length(diffusion, lifetime_s) * 1e4
    x_um = np.linspace(0, max(100, 6 * length_um), 500)
    carriers = photodiode.excess_carrier_profile(
        x_um,
        diffusion_cm2_s=diffusion,
        lifetime_s=lifetime_s,
        junction_excess_cm3=junction,
        generation_cm3_s=generation,
    )
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.semilogy(x_um, carriers, color="#7b1e24", linewidth=2.5)
    ax.set(xlabel="Distance from junction (um)", ylabel="Excess carriers (cm-3)")
    ax.set_title(f"Equation 3.11: L_e = {length_um:.1f} um")
    plt.show()

## Equations 3.14, 3.19, 3.22, and 3.27

This overview generates the current-voltage family, ideal spectral cutoff, absorption curve, and responsivity curve from the engine. Change the values in the first six lines and run the cell again.

In [ ]:
temperature_k = 300.0
ideality_factor = 1.4
generation_cm3_s = (0.0, 1e11, 3e11)
bandgap_ev = 1.12
quantum_efficiency = 0.80
absorption_cm_inv = 100.0

parameters = photodiode.PhotodiodeParameters(
    temperature_k=temperature_k,
    ideality_factor=ideality_factor,
)
voltage = np.linspace(-0.5, 0.5, 500)
wavelength = np.linspace(0.3, 1.5, 500)
depth_um = np.linspace(0, 500, 500)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for generation in generation_cm3_s:
    current = photodiode.photodiode_current_density(
        voltage, parameters=parameters, generation_cm3_s=generation
    )
    axes[0, 0].plot(voltage, current, label=f"G_L={generation:.0e}")
axes[0, 0].set(xlabel="Voltage (V)", ylabel="Current density (A/cm2)", title="Equation 3.14")
axes[0, 0].legend()

axes[0, 1].plot(
    wavelength,
    photodiode.ideal_spectral_response(wavelength, bandgap_ev),
    color="#7b1e24",
)
axes[0, 1].set(xlabel="Wavelength (um)", ylabel="Spectral response", title="Equation 3.19")

axes[1, 0].plot(
    depth_um,
    photodiode.absorption_intensity(depth_um, absorption_cm_inv),
    color="#287271",
)
axes[1, 0].set(xlabel="Depth (um)", ylabel="I / I_0", title="Equation 3.22")

axes[1, 1].plot(
    wavelength,
    photodiode.responsivity(wavelength, quantum_efficiency, bandgap_ev),
    color="#c18b2e",
)
axes[1, 1].set(xlabel="Wavelength (um)", ylabel="Responsivity (A/W)", title="Equation 3.27")

fig.tight_layout()
plt.show()

## Section 3.4.1: silicon absorption and surface reflection

For a fixed beam area, power follows the same Beer-Lambert law as intensity. The green curve assumes that all source power enters the silicon. The red curve first applies the normal-incidence air-to-silicon Fresnel loss, then applies the same bulk absorption: $P(x)=(1-R)P_0e^{-\alpha x}$.

In [ ]:
@interact(
    incident_power_w=FloatLogSlider(value=0.1, base=10, min=-3, max=1, step=0.02, description="P0 (W)"),
    absorption_cm_inv=FloatLogSlider(value=100, base=10, min=2, max=5, step=0.05, description="alpha (cm-1)"),
    silicon_index=FloatSlider(value=3.5, min=3.2, max=4.2, step=0.01, description="n silicon"),
    displayed_lengths=FloatSlider(value=5, min=1, max=8, step=0.1, description="depth (1/alpha)"),
)
def plot_silicon_absorption(
    incident_power_w, absorption_cm_inv, silicon_index, displayed_lengths
):
    absorption_length_um = 1e4 / absorption_cm_inv
    depth_um = np.linspace(0, displayed_lengths * absorption_length_um, 600)
    reflectance = photodiode.fresnel_reflectance(1.0, silicon_index)
    no_surface_loss = photodiode.absorption_power(
        depth_um, absorption_cm_inv, incident_power_w
    )
    with_surface_loss = photodiode.absorption_power(
        depth_um,
        absorption_cm_inv,
        incident_power_w,
        surface_reflectance=reflectance,
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(depth_um, no_surface_loss, label="no surface reflection", color="#287271", linewidth=2.5)
    ax.plot(depth_um, with_surface_loss, label="air-to-silicon surface", color="#7b1e24", linewidth=2.5)
    ax.set(xlabel="Depth inside silicon (um)", ylabel="Optical power remaining (W)")
    ax.set_title(
        f"R = {100 * reflectance:.1f}%,  1/alpha = {absorption_length_um:.1f} um"
    )
    ax.set_ylim(bottom=0)
    ax.legend()
    plt.show()

## Figure 3.10: single-layer antireflection coating

The chapter gives the ideal index and quarter-wave thickness. The engine evaluates the full wavelength-dependent interference of one lossless film.

In [ ]:
@interact(
    design_um=FloatSlider(value=1.0, min=0.45, max=1.5, step=0.01, description="lambda_0"),
    substrate_index=FloatSlider(value=3.5, min=1.5, max=4.5, step=0.01, description="n substrate"),
    film_index=FloatSlider(value=1.87, min=1.1, max=3.0, step=0.01, description="n film"),
    thickness_factor=FloatSlider(value=1.0, min=0.4, max=1.6, step=0.01, description="t / t_qw"),
)
def plot_coating(design_um, substrate_index, film_index, thickness_factor):
    wavelength_um = np.linspace(0.35, 1.75, 600)
    quarter_wave_um = design_um / (4 * film_index)
    coated_r = photodiode.single_layer_reflectance(
        wavelength_um,
        index_incident=1.0,
        index_film=film_index,
        index_substrate=substrate_index,
        thickness_um=quarter_wave_um * thickness_factor,
    )
    uncoated_r = photodiode.fresnel_reflectance(1.0, substrate_index)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(wavelength_um, 100 * (1 - coated_r), label="coated", color="#7b1e24")
    ax.axhline(100 * (1 - uncoated_r), label="uncoated", color="#c18b2e")
    ax.set(xlabel="Wavelength (um)", ylabel="Reflection-limited efficiency (%)")
    ax.legend()
    plt.show()